# IT ranking plots — standalone
Loads pre-computed results from disk (no training). Run while the IT_AT notebook is still running AT.

In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import pandas as pd

sys.path.append(os.path.join(os.getcwd(), '../../'))

import massbalancemachine as mbm
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.plots.use_mbm_style()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Paths

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / 'TF_REGION'
CACHE_DIR = BASE_DIR / 'LSTM_cache_TL/Transformer_exp_IT_AT'
CACHE_DIR_RANK_ITAT = CACHE_DIR / 'ranking_itat'

os.makedirs(BASE_DIR / 'figures', exist_ok=True)

print('BASE_DIR :', BASE_DIR)
print('CACHE_DIR:', CACHE_DIR)

## Constants (must match the training notebook)

In [ ]:
MONTHLY_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
STATIC_COLS = ['aspect', 'slope', 'svf']

ALL_REGIONS = ['CH', 'NOR', 'ISL', 'FR']
FRACS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.4]
N_RANDOM_SEEDS = 10
FRACS_TO_PLOT = [10, 15]
MODEL_DATE = '2026-05-21'

IT_GLACIERS = [
    'CAMPO SETT.',
    'CARESER',
    'CARESER CENTRALE',
    'CARESER OCCIDENTALE',
    'CARESER ORIENTALE',
    'CIARDONEY',
    'FONTANA BIANCA / WEISSBRUNNF.',
    'GRAND ETRET',
    'LUNGA (VEDRETTA) / LANGENF.',
    'LUPO',
    'MALAVALLE (VEDR. DI) / UEBELTALF.',
    'PENDENTE (VEDR.) / HANGENDERF.',
    'RIES OCC. (VEDR. DI) / RIESERF. WESTL.',
    'SURETTA MERIDIONALE',
]

## Load best Transformer params from GS log

In [ ]:
gs_logs_dir = BASE_DIR / 'logs/Transformer_GS'
logs_gs = pd.read_csv(gs_logs_dir / 'transformer_gs_pooled_2026-05-20.csv')
best_row = logs_gs.sort_values('valid_loss').iloc[0]

best_params_gs = {
    'Fm':
    int(best_row['Fm']),
    'Fs':
    int(best_row['Fs']),
    'd_model':
    int(best_row['d_model']),
    'nhead':
    int(best_row['nhead']),
    'num_layers':
    int(best_row['num_layers']),
    'dim_feedforward':
    int(best_row['dim_feedforward']),
    'dropout':
    float(best_row['dropout']),
    'static_layers':
    int(best_row['static_layers']),
    'static_hidden': (None if pd.isna(best_row['static_hidden']) else int(
        best_row['static_hidden'])),
    'static_dropout': (None if pd.isna(best_row['static_dropout']) else float(
        best_row['static_dropout'])),
    'head_dropout':
    float(best_row['head_dropout']),
    'lr':
    float(best_row['lr']),
    'weight_decay':
    float(best_row['weight_decay']),
    'two_heads':
    False,
    'loss_spec':
    None,
    'T_max':
    32,
}
print('best_params_gs loaded')

## Load IT ranking CSVs

In [ ]:
df_ranked_it_joint = pd.read_csv(BASE_DIR / 'glacier_ranking_it_joint.csv')
df_ranked_it_topo = pd.read_csv(BASE_DIR / 'glacier_ranking_it_topo.csv')
print('IT joint top-5:')
print(df_ranked_it_joint.head())
print('\nIT topo  top-5:')
print(df_ranked_it_topo.head())

## Rebuild IT glacier subsets (deterministic — matches training run)

In [ ]:
gl_subsets_it_joint = build_glacier_subsets(
    df_ranked=df_ranked_it_joint,
    fracs=FRACS,
    n_random_seeds=N_RANDOM_SEEDS,
    seed=cfg.seed,
)
gl_subsets_it_topo = build_glacier_subsets(
    df_ranked=df_ranked_it_topo,
    fracs=FRACS,
    n_random_seeds=N_RANDOM_SEEDS,
    seed=cfg.seed,
)
print('Glacier subsets rebuilt')

## Load results CSV and build IT dataset

In [ ]:
# Reload monthly data to build the IT test DataLoader
paths_multi = {
    'EU_US_CANADA': {
        'era5_climate_data':
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     'era5_monthly_averaged_data_EU_US_CANADA.nc'),
        'geopotential_data':
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     'era5_geopotential_pressure_EU_US_CANADA.nc'),
    },
}
vois_climate = ['t2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str']
vois_topographical = ['aspect', 'slope', 'svf']

dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

run_flag_by_code = {
    c: False
    for c in ['CH', 'NOR', 'ISL', 'FR', 'SJM', 'IT_AT']
}
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=ALL_REGIONS + ['IT_AT'],
    run_flag_by_code=run_flag_by_code,
)

res_xreg_by_source = {r: monthly_cache[r] for r in ALL_REGIONS + ['IT_AT']}

df_itat_full = res_xreg_by_source['IT_AT']['data_monthly']
df_itat_aug = res_xreg_by_source['IT_AT']['data_monthly_aug']

months_head_pad = res_xreg_by_source['CH']['months_head_pad']
months_tail_pad = res_xreg_by_source['CH']['months_tail_pad']

ds_it = build_combined_LSTM_dataset(
    df_loss=df_itat_full[df_itat_full['GLACIER'].isin(IT_GLACIERS)],
    df_full=df_itat_aug[df_itat_aug['GLACIER'].isin(IT_GLACIERS)],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)
print(f'IT test set: {len(ds_it)} sequences')

In [ ]:
# Re-evaluate all IT models to rebuild df_ranking_results_itat in memory.
# The original notebook writes the CSV only after AT is done, so we reconstruct it here.
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', 'FR']

lstm_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None,
}

ranking_results_it = []

for ranking_label, df_ranked, gl_subsets_cur in [
    ('ranked_by_IT_joint', df_ranked_it_joint, gl_subsets_it_joint),
    ('ranked_by_IT_topo', df_ranked_it_topo, gl_subsets_it_topo),
]:
    print(f"\n{'='*60}")
    print(f'  Ranking: {ranking_label}  →  eval on IT')

    mode = ranking_label.split('_')[-1]
    models_dir = BASE_DIR / 'models/ranking_itat' / mode

    # --- per-fraction subsets ---
    for pct, subset in gl_subsets_cur.items():
        strategies = {
            'closest': subset['closest'],
            **{
                f'random_{s["seed_idx"]}': s['glaciers']
                for s in subset['random']
            },
        }
        for strategy_name, glaciers in strategies.items():
            assets = build_assets_from_glacier_list(
                glaciers=glaciers,
                df_ranked=df_ranked,
                res_xreg_by_source=res_xreg_by_source,
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                cfg=cfg,
                cache_path=str(
                    CACHE_DIR_RANK_ITAT /
                    f'assets_IT_{mode}_{pct}pct_{strategy_name}.joblib'),
                force_recompute=False,
                months_head_pad=months_head_pad,
                months_tail_pad=months_tail_pad,
                src_regions=SOURCE_REGIONS,
            )
            for model_type, params, model_label in [
                ('transformer', best_params_gs, 'transformer'),
                ('lstm', lstm_params, 'lstm'),
            ]:
                model, _, _ = train_or_load_one_source_model(
                    cfg=cfg,
                    key=f'IT_{pct}pct_{strategy_name}',
                    lstm_assets=assets,
                    best_params=params,
                    device=device,
                    models_dir=models_dir,
                    prefix=f'{model_type}_rank_itat',
                    train_flag=False,
                    force_retrain=False,
                    epochs=150,
                    date=MODEL_DATE,
                    model_type=model_type,
                    verbose=False,
                )
                ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                    assets['ds_train'])
                ds_scaler.make_loaders(
                    train_idx=assets['train_idx'],
                    val_idx=assets['val_idx'],
                    fit_and_transform=True,
                    seed=cfg.seed,
                    verbose=False,
                )
                ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                    ds_it)
                test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                    ds_copy,
                    ds_scaler,
                    batch_size=128,
                    seed=cfg.seed,
                )
                metrics, _ = model.evaluate_with_preds(device, test_dl,
                                                       ds_copy)
                is_random = strategy_name.startswith('random')
                ranking_results_it.append({
                    'ranking_target':
                    ranking_label,
                    'model':
                    model_label,
                    'pct':
                    pct,
                    'strategy':
                    'random' if is_random else strategy_name,
                    'seed_idx':
                    int(strategy_name.split('_')[1]) if is_random else None,
                    'test_region':
                    'IT',
                    'n_glaciers':
                    len(glaciers),
                    'n_sequences':
                    len(assets['ds_train']),
                    **{
                        k: round(v, 3)
                        for k, v in metrics.items()
                    },
                })

    # --- 100 % baseline ---
    full_assets = build_assets_from_glacier_list(
        glaciers=df_ranked['glacier'].tolist(),
        df_ranked=df_ranked,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(CACHE_DIR_RANK_ITAT /
                       f'assets_IT_{mode}_100pct_full.joblib'),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=SOURCE_REGIONS,
    )
    for model_type, params, label in [
        ('lstm', lstm_params, 'lstm_full'),
        ('transformer', best_params_gs, 'transformer_full'),
    ]:
        model, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key='IT_100pct_full',
            lstm_assets=full_assets,
            best_params=params,
            device=device,
            models_dir=models_dir,
            prefix=f'{model_type}_rank_itat',
            train_flag=False,
            force_retrain=False,
            epochs=150,
            date=MODEL_DATE,
            model_type=model_type,
            verbose=False,
        )
        ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            full_assets['ds_train'])
        ds_scaler.make_loaders(
            train_idx=full_assets['train_idx'],
            val_idx=full_assets['val_idx'],
            fit_and_transform=True,
            seed=cfg.seed,
            verbose=False,
        )
        ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            ds_it)
        test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_copy,
            ds_scaler,
            batch_size=128,
            seed=cfg.seed,
        )
        metrics, _ = model.evaluate_with_preds(device, test_dl, ds_copy)
        ranking_results_it.append({
            'ranking_target': ranking_label,
            'model': label.replace('_full', ''),
            'pct': 100,
            'strategy': label,
            'seed_idx': None,
            'test_region': 'IT',
            'n_glaciers': len(df_ranked),
            'n_sequences': len(full_assets['ds_train']),
            **{
                k: round(v, 3)
                for k, v in metrics.items()
            },
        })

df_ranking_results_itat = pd.DataFrame(ranking_results_it)
print(
    df_ranking_results_itat.groupby(
        ['ranking_target', 'model', 'pct',
         'strategy']).mean(numeric_only=True).round(3))

## Plot 1 — RMSE vs fraction curves (IT only)

In [ ]:
# --- four plots ---
for model_label in ["transformer", "lstm"]:
    for ranking_label in ['ranked_by_IT_joint', 'ranked_by_IT_topo']:
        fig = plot_ranking_results_extended(
            df_results=df_ranking_results_itat,
            ranking_label=ranking_label,
            test_region="IT",
            source_regions=ALL_REGIONS,
            n_rand_seeds=N_RANDOM_SEEDS,
            model_label=model_label,
            save_path=f"ranking_it_{ranking_label}_{model_label}",
        )

## Plot 2 — pred vs truth grid (IT, fracs 10 % & 15 %)

In [ ]:
for ranking_label, df_ranked, gl_subsets_cur in [
    ('ranked_by_IT_joint', df_ranked_it_joint, gl_subsets_it_joint),
    ('ranked_by_IT_topo', df_ranked_it_topo, gl_subsets_it_topo),
]:
    mode = ranking_label.split('_')[-1]  # 'joint' or 'topo'
    models_dir = BASE_DIR / 'models/ranking_itat' / mode
    ranking_plot_configs = []

    for pct in FRACS_TO_PLOT:

        # --- closest ---
        assets_close = build_assets_from_glacier_list(
            glaciers=gl_subsets_cur[pct]['closest'],
            df_ranked=df_ranked,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=str(CACHE_DIR_RANK_ITAT /
                           f'assets_IT_{mode}_{pct}pct_closest.joblib'),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
        )
        model_close, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=f'IT_{pct}pct_closest',
            lstm_assets=assets_close,
            best_params=best_params_gs,
            device=device,
            models_dir=models_dir,
            prefix='transformer_rank_itat',
            train_flag=False,
            date=MODEL_DATE,
            model_type='transformer',
            verbose=False,
        )
        ranking_plot_configs.append(
            (f'Closest {pct}% ({mode})', model_close, assets_close))

        # --- random seed 0 ---
        assets_rnd = build_assets_from_glacier_list(
            glaciers=gl_subsets_cur[pct]['random'][0]['glaciers'],
            df_ranked=df_ranked,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=str(CACHE_DIR_RANK_ITAT /
                           f'assets_IT_{mode}_{pct}pct_random_0.joblib'),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
        )
        model_rnd, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=f'IT_{pct}pct_random_0',
            lstm_assets=assets_rnd,
            best_params=best_params_gs,
            device=device,
            models_dir=models_dir,
            prefix='transformer_rank_itat',
            train_flag=False,
            date=MODEL_DATE,
            model_type='transformer',
            verbose=False,
        )
        ranking_plot_configs.append(
            (f'Random {pct}% seed 0 ({mode})', model_rnd, assets_rnd))

    plot_pred_vs_truth_grid(
        plot_configs=ranking_plot_configs,
        ds_test=ds_it,
        device=device,
        cfg=cfg,
        ncols=4,
        ax_xlim=(-8, 6),
        ax_ylim=(-8, 6),
        figsize=(25, 12),
        title=f'Closest vs random ({ranking_label}) → IT',
        save_path=BASE_DIR /
        f'figures/ranking_itat_{ranking_label}_pred_vs_truth_IT',
    )

In [ ]:
label, model, region_assets =ranking_plot_configs[0]
ds_scaler_copy = (
            mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                region_assets["ds_train"]
            )
        )
ds_scaler_copy.make_loaders(
    train_idx=region_assets["train_idx"],
    val_idx=region_assets["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

ds_test_copy = (
    mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(ds_it)
)
test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
    ds_test_copy,
    ds_scaler_copy,
    batch_size=128,
    seed=cfg.seed,
)

_, df_preds = model.evaluate_with_preds(device, test_dl, ds_test_copy)
df_preds = df_preds.dropna(subset=["target", "pred"])

scores_annual, scores_winter = compute_seasonal_scores(
    df_preds, target_col="target", pred_col="pred"
)
df_preds

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

glaciers = df_preds["GLACIER"].unique()
cmap     = cm.get_cmap("tab20", len(glaciers))
colors   = {g: cmap(i) for i, g in enumerate(glaciers)}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, season in zip(axes, ["annual", "winter"]):
    mask = df_preds["PERIOD"] == season   # adjust column/values as needed
    df_s = df_preds[mask]

    for glacier, grp in df_s.groupby("GLACIER"):
        ax.scatter(
            grp["target"], grp["pred"],
            color=colors[glacier], label=glacier, s=30, alpha=0.8,
        )

    lim = [df_preds[["target", "pred"]].min().min() - 0.2,
           df_preds[["target", "pred"]].max().max() + 0.2]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Observed (m w.e.)")
    ax.set_ylabel("Predicted (m w.e.)")
    ax.set_title(season.capitalize())

# single legend outside the plots
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right",
           bbox_to_anchor=(1.15, 0.5), fontsize=8, frameon=False)
fig.suptitle(label, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df_preds[df_preds.GLACIER == 'LUPO'].plot(x='target', y='pred', kind='scatter')

In [ ]:
df_preds[df_preds.GLACIER == 'LUPO']